In [ ]:
# 在我混乱的文件夹里把sorting结果翻出来排列好一个个打开真是救命了尤其数据难看得要命

from pathlib import Path

root = Path(r"C:\Users\hetin\Documents\project1\ephys-data")

kilosort4_dirs = [p for p in root.rglob('kilosort4') if p.is_dir()]
for p in kilosort4_dirs:
    print(p)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec, rcParams

def load_kilosort_data(results_dir):
    results_dir = Path(results_dir)
    
    ops          = np.load(results_dir / 'ops.npy', allow_pickle=True).item()
    camps        = pd.read_csv(results_dir / 'cluster_Amplitude.tsv', sep='\t')['Amplitude'].values
    contam_pct   = pd.read_csv(results_dir / 'cluster_ContamPct.tsv', sep='\t')['ContamPct'].values
    chan_map      = np.load(results_dir / 'channel_map.npy')
    templates     = np.load(results_dir / 'templates.npy')
    chan_best_idx = (templates**2).sum(axis=1).argmax(axis=-1)  # 用于切片 templates
    chan_best     = chan_map[chan_best_idx]                       # 用于画图坐标
    amplitudes    = np.load(results_dir / 'amplitudes.npy')
    st            = np.load(results_dir / 'spike_times.npy')
    clu           = np.load(results_dir / 'spike_clusters.npy')
    firing_rates  = np.unique(clu, return_counts=True)[1] * 30000 / st.max()
    dshift        = ops['dshift']
    
    return ops, camps, contam_pct, chan_map, chan_best, chan_best_idx, amplitudes, st, clu, firing_rates, dshift, templates


def plot_kilosort_summary(results_dir):
    """Load data and plot Kilosort4 sorting summary."""
    ops, camps, contam_pct, chan_map, chan_best, chan_best_idx, amplitudes, st, clu, firing_rates, dshift, templates = \
    load_kilosort_data(kilosort4_dirs[0])


    gray = .5 * np.ones(3)
    fig = plt.figure(figsize=(10,10), dpi=100)
    grid = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.5)

    ax = fig.add_subplot(grid[0,0])
    ax.plot(np.arange(0, ops['Nbatches'])*2, dshift)
    ax.set_xlabel('time (sec.)')
    ax.set_ylabel('drift (um)')

    ax = fig.add_subplot(grid[0,1:])
    t0 = 0
    t1 = np.nonzero(st > ops['fs']*5)[0][0]
    ax.scatter(st[t0:t1]/30000., chan_best[clu[t0:t1]], s=0.5, color='k', alpha=0.25)
    ax.set_xlim([0, 5])
    ax.set_ylim([chan_map.max(), 0])
    ax.set_xlabel('time (sec.)')
    ax.set_ylabel('channel')
    ax.set_title('spikes from units')

    ax = fig.add_subplot(grid[1,0])
    ax.hist(firing_rates, 20, color=gray)
    ax.set_xlabel('firing rate (Hz)')
    ax.set_ylabel('# of units')

    ax = fig.add_subplot(grid[1,1])
    ax.hist(camps, 20, color=gray)
    ax.set_xlabel('amplitude')
    ax.set_ylabel('# of units')

    ax = fig.add_subplot(grid[1,2])
    nb = ax.hist(np.minimum(100, contam_pct), np.arange(0,105,5), color=gray)
    ax.plot([10, 10], [0, nb[0].max()], 'k--')
    ax.set_xlabel('% contamination')
    ax.set_ylabel('# of units')
    ax.set_title('< 10% = good units')

    is_ref = contam_pct < 10.
    for k in range(2):
        ax = fig.add_subplot(grid[2,k])
        ax.scatter(firing_rates[~is_ref], camps[~is_ref], s=3, color='r', label='mua', alpha=0.25)
        ax.scatter(firing_rates[is_ref],  camps[is_ref],  s=3, color='b', label='good', alpha=0.25)
        ax.set_ylabel('amplitude (a.u.)')
        ax.set_xlabel('firing rate (Hz)')
        ax.legend()
        if k == 1:
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.set_title('loglog')

    plt.suptitle(str(results_dir), fontsize=8, y=1.01)
  
    return fig


def plot_unit_waveforms(ops, templates, chan_best, clu, contam_pct, 
                        n_random=40, nc=16, amp=4, 
                        label=None, title=None):
    """
    Plot random unit waveforms for good and mua units.

    Parameters
    ----------
    ops         : dict, from load_kilosort_data
    templates   : np.array, templates
    chan_best   : np.array, best channel per unit
    clu         : np.array, spike cluster labels
    contam_pct  : np.array, contamination percentage per unit
    n_random    : int, number of random units to show (default 40)
    nc          : int, number of channels to show around best channel (default 16)
    amp         : float, waveform amplitude scaling (default 4)
    label       : 'good', 'mua', or None (default None = plot both)
    title       : str, overall figure title (default None)
    """
    probe = ops['probe']
    xc, yc = probe['xc'], probe['yc']

    good_units = np.nonzero(contam_pct <= 0.1)[0]
    mua_units  = np.nonzero(contam_pct >  0.1)[0]

    all_groups = [('good', good_units), ('mua', mua_units)]

    # 根据 label 参数过滤
    if label == 'good':
        groups = [('good', good_units)]
    elif label == 'mua':
        groups = [('mua', mua_units)]
    elif label is None:
        groups = all_groups
    else:
        raise ValueError(f"label 只能是 'good', 'mua', 或 None，不能是 '{label}'")

    figs = []
    for gstr, units in groups:
        if len(units) == 0:
            print(f'No {gstr} units found, skipping.')
            continue

        n_cols = 20
        n_rows = 2
        n_plot = min(n_random, len(units))

        fig = plt.figure(figsize=(12, 3), dpi=150)
        grid = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.25, wspace=0.5)

        # 图标题：优先用传入的 title，否则用 gstr
        fig_title = f'{title} — {gstr}' if title else gstr
        fig.suptitle(fig_title, fontsize=10, fontweight='bold')

        for k in range(n_plot):
            wi = units[np.random.randint(len(units))]
            wv = templates[wi].copy()
            cb = chan_best[wi]
            nsp = (clu == wi).sum()

            ax = fig.add_subplot(grid[k // n_cols, k % n_cols])
            n_chan = wv.shape[-1]
            ic0 = max(0, cb - nc // 2)
            ic1 = min(n_chan, cb + nc // 2)
            wv = wv[:, ic0:ic1]
            x0, y0 = xc[ic0:ic1], yc[ic0:ic1]

            for ii, (xi, yi) in enumerate(zip(x0, y0)):
                t = np.arange(-wv.shape[0]//2, wv.shape[0]//2, 1, 'float32')
                t /= wv.shape[0] / 20
                ax.plot(xi + t, yi + wv[:, ii] * amp, lw=0.5, color='k')

            ax.set_title(f'{nsp}', fontsize='small')
            ax.axis('off')

        figs.append(fig)
        plt.close()

    return figs

In [ ]:
ops, camps, contam_pct, chan_map, chan_best, chan_best_idx, amplitudes, st, clu, firing_rates, dshift, templates = \
    load_kilosort_data(kilosort4_dirs[0])
figs = plot_unit_waveforms(ops, templates, chan_best, clu, contam_pct)
for fig in figs:
    display(fig)

In [ ]:
fig = plot_kilosort_summary(kilosort4_dirs[0])

In [ ]:
def plot_all_kilosort_summary(kilosort4_dirs):
    """Plot Kilosort4 summary for all directories in one big figure."""
    
    # 先过滤掉缺文件的文件夹
    valid_dirs = []
    for d in kilosort4_dirs:
        try:
            load_kilosort_data(d)
            valid_dirs.append(d)
        except FileNotFoundError as e:
            print(f'跳过 {d}：{e}')

    if len(valid_dirs) == 0:
        print('没有有效的 kilosort4 文件夹')
        return None

    n = len(valid_dirs)
    fig = plt.figure(figsize=(30, 10 * n), dpi=100)
    outer_grid = gridspec.GridSpec(n, 1, figure=fig, hspace=0.7)

    for idx, results_dir in enumerate(valid_dirs):
        ops, camps, contam_pct, chan_map, chan_best, amplitudes, st, clu, firing_rates, dshift = \
            load_kilosort_data(results_dir)

        inner_grid = gridspec.GridSpecFromSubplotSpec(
            3, 3, subplot_spec=outer_grid[idx], hspace=0.5, wspace=0.5
        )

        # 标题
        ax_title = fig.add_subplot(outer_grid[idx])
        ax_title.set_title(str(results_dir), fontsize=10, fontweight='bold', pad=40)
        ax_title.axis('off')

        gray = .5 * np.ones(3)

        # drift
        ax = fig.add_subplot(inner_grid[0, 0])
        ax.plot(np.arange(0, ops['Nbatches']) * 2, dshift)
        ax.set_xlabel('time (sec.)')
        ax.set_ylabel('drift (um)')

        # spikes raster
        ax = fig.add_subplot(inner_grid[0, 1:])
        t0 = 0
        t1 = np.nonzero(st > ops['fs'] * 5)[0][0]
        ax.scatter(st[t0:t1] / 30000., chan_best[clu[t0:t1]], s=0.5, color='k', alpha=0.25)
        ax.set_xlim([0, 5])
        ax.set_ylim([chan_map.max(), 0])
        ax.set_xlabel('time (sec.)')
        ax.set_ylabel('channel')
        ax.set_title('spikes from units')

        # firing rate
        ax = fig.add_subplot(inner_grid[1, 0])
        ax.hist(firing_rates, 20, color=gray)
        ax.set_xlabel('firing rate (Hz)')
        ax.set_ylabel('# of units')

        # amplitude
        ax = fig.add_subplot(inner_grid[1, 1])
        ax.hist(camps, 20, color=gray)
        ax.set_xlabel('amplitude')
        ax.set_ylabel('# of units')

        # contamination
        ax = fig.add_subplot(inner_grid[1, 2])
        nb = ax.hist(np.minimum(100, contam_pct), np.arange(0, 105, 5), color=gray)
        ax.plot([10, 10], [0, nb[0].max()], 'k--')
        ax.set_xlabel('% contamination')
        ax.set_ylabel('# of units')
        ax.set_title('< 10% = good units')

        # scatter
        is_ref = contam_pct < 10.
        for k in range(2):
            ax = fig.add_subplot(inner_grid[2, k])
            ax.scatter(firing_rates[~is_ref], camps[~is_ref], s=3, color='r', label='mua', alpha=0.25)
            ax.scatter(firing_rates[is_ref],  camps[is_ref],  s=3, color='b', label='good', alpha=0.25)
            ax.set_ylabel('amplitude (a.u.)')
            ax.set_xlabel('firing rate (Hz)')
            ax.legend()
            if k == 1:
                ax.set_xscale('log')
                ax.set_yscale('log')
                ax.set_title('loglog')

    plt.close()
    return fig


In [ ]:
# 调用
fig = plot_all_kilosort_summary(kilosort4_dirs)
display(fig)

# 保存
#fig.savefig('kilosort_summary_all.pdf', bbox_inches='tight')

In [ ]:
def plot_all_unit_waveforms(kilosort4_dirs, n_random=40, nc=16, amp=4, label=None):
    """
    Plot unit waveforms for all kilosort4 directories on a single figure.
    
    Parameters
    ----------
    kilosort4_dirs : list of Path, list of kilosort4 directories
    n_random       : int, number of random units to show per group (default 40)
    nc             : int, number of channels to show around best channel (default 16)
    amp            : float, waveform amplitude scaling (default 4)
    label          : 'good', 'mua', or None (default None = plot both)
    """
    # 过滤有效目录
    valid_data = []
    for d in kilosort4_dirs:
        try:
            data = load_kilosort_data(d)
            valid_data.append((d, data))
        except FileNotFoundError as e:
            print(f'跳过 {d}：{e}')

    if not valid_data:
        print('没有有效的 kilosort4 文件夹')
        return None

    # 计算总行数
    if label is None:
        n_groups_per_dir = 2
    else:
        n_groups_per_dir = 1
    
    n_dirs = len(valid_data)
    n_cols = 20
    n_rows_per_group = 2

    fig = plt.figure(figsize=(12, 3 * n_groups_per_dir * n_dirs), dpi=150)
    outer_grid = gridspec.GridSpec(n_dirs, 1, figure=fig, hspace=0.8)

    for dir_idx, (d, data) in enumerate(valid_data):
        ops, camps, contam_pct, chan_map, chan_best, chan_best_idx, amplitudes, st, clu, firing_rates, dshift, templates = data

        probe = ops['probe']
        xc, yc = probe['xc'], probe['yc']

        good_units = np.nonzero(contam_pct <= 0.1)[0]
        mua_units  = np.nonzero(contam_pct >  0.1)[0]

        if label == 'good':
            groups = [('good', good_units)]
        elif label == 'mua':
            groups = [('mua', mua_units)]
        else:
            groups = [('good', good_units), ('mua', mua_units)]

        groups = [(g, u) for g, u in groups if len(u) > 0]

        # 每个 dir 的子 grid
        dir_grid = gridspec.GridSpecFromSubplotSpec(
            len(groups), 1,
            subplot_spec=outer_grid[dir_idx],
            hspace=0.6
        )

        # dir 标题
        ax_dir = fig.add_subplot(outer_grid[dir_idx])
        ax_dir.set_title(str(d), fontsize=9, fontweight='bold', loc='left', pad=40)
        ax_dir.axis('off')

        for group_idx, (gstr, units) in enumerate(groups):
            n_plot = min(n_random, len(units))

            inner_grid = gridspec.GridSpecFromSubplotSpec(
                n_rows_per_group, n_cols,
                subplot_spec=dir_grid[group_idx],
                hspace=0.25, wspace=0.5
            )

            # group 标题
            ax_label = fig.add_subplot(dir_grid[group_idx])
            ax_label.set_title(f'{gstr} units  (n={len(units)})',
                               fontsize=8, loc='left', pad=15)
            ax_label.axis('off')

            for k in range(n_plot):
                wi = units[np.random.randint(len(units))]
                wv = templates[wi].copy()
                cb = chan_best_idx[wi]
                nsp = (clu == wi).sum()

                ax = fig.add_subplot(inner_grid[k // n_cols, k % n_cols])
                n_chan = wv.shape[-1]
                ic0 = max(0, cb - nc // 2)
                ic1 = min(n_chan, cb + nc // 2)
                wv = wv[:, ic0:ic1]
                x0, y0 = xc[ic0:ic1], yc[ic0:ic1]

                for ii, (xi, yi) in enumerate(zip(x0, y0)):
                    t = np.arange(-wv.shape[0]//2, wv.shape[0]//2, 1, 'float32')
                    t /= wv.shape[0] / 20
                    ax.plot(xi + t, yi + wv[:, ii] * amp, lw=0.5, color='k')

                ax.set_title(f'{nsp}', fontsize='small')
                ax.axis('off')

    plt.close()
    return fig

In [ ]:
fig = plot_all_unit_waveforms(kilosort4_dirs)
display(fig)

original with neuropixel demo data:
https://colab.research.google.com/github/mouseland/kilosort/blob/main/docs/tutorials/kilosort4.ipynb#scrollTo=qP5nzDMbjiO8